# Практическая работа №1
## «Numpy и основные понятия в аналитике»

**Вариант 2.** Шагомер (количество шагов за 30 дней)

**Цель работы:** освоить основные функции библиотеки NumPy для расчёта показателей описательной статистики, научиться находить выбросы с помощью межквартильного размаха и выполнять операции с матрицами.

## Ход работы
### Часть 1. Анализ массива data_steps

Для начала подключаем библиотеку NumPy и создаём массив `data_steps` с исходными данными: количеством шагов за 30 дней.

In [1]:
import numpy as np

In [2]:
data_steps = np.array([
8200, 11500, 9400, 7100, 10800, 8600, 12900, 45000, 9900, 9100,
7600, 11300, 8300, 9700, 10400, 8900, 12100, 6800, 10200, 9200,
200, 450, 8100, 11600, 7900, 9600, 10700, 8700, 12200, 6900
])

Считаем основные показатели: среднее значение, медиану, максимальное и минимальное значения, дисперсию, стандартное отклонение и размах. Дисперсия и стандартное отклонение считаются как выборочные (`ddof=1`).

In [3]:
print('Ср знач: ', np.mean(data_steps))
print('Медиана: ', np.median(data_steps))
print('Максимальное значение: ', np.max(data_steps))
print('Минимальное значение: ', np.min(data_steps))
print('Дисперсия: ', np.var(data_steps, ddof=1))
print('Стандартное отклонение: ',  np.std(data_steps, ddof=1))
print('Размах: ', np.ptp(data_steps))

Ср знач:  10111.666666666666
Медиана:  9300.0
Максимальное значение:  45000
Минимальное значение:  200
Дисперсия:  51464083.333333336
Стандартное отклонение:  7173.847178002424
Размах:  44800


Среднее заметно больше медианы, а стандартное отклонение и размах очень большие: это уже указывает на наличие выбросов.

Далее считаем 25, 50, 75, 90 и 99 перцентили, первый и третий квартили и межквартильный размах.

In [4]:
percentiles = [25, 50, 75, 90, 99]

In [5]:
for p in percentiles:
  print(f'{p}-й перцентиль {np.percentile(data_steps, p)}')

25-й перцентиль 8125.0
50-й перцентиль 9300.0
75-й перцентиль 10775.0
90-й перцентиль 12110.0
99-й перцентиль 35691.00000000003


In [6]:
q1 = np.percentile(data_steps, 25)
q3 = np.percentile(data_steps, 75)
print(q1, q3)

8125.0 10775.0


In [7]:
iqr = q3 - q1
print('межквартильный размах: ', iqr)

межквартильный размах:  2650.0


99-й перцентиль резко выше 90-го, что также говорит о наличии аномально больших значений.

С помощью межквартильного размаха находим границы нормы: Q1 − 1,5·IQR и Q3 + 1,5·IQR. Значения за этими границами считаются выбросами.

In [8]:
low = q1-1.5*iqr
high = q3 + 1.5*iqr
outliers = data_steps[(data_steps < low) | (data_steps > high)]
print("Выбросы:", outliers)
print('границы нормы: ', low, 'и', high)

Выбросы: [45000   200   450]
границы нормы:  4150.0 и 14750.0


Чтобы оценить влияние выбросов, сравниваем среднее значение с выбросами и без них.

In [9]:
clean = data_steps[(data_steps >= low) & (data_steps <= high)]
print('Среднее с выбросами: ', np.mean(data_steps))
print('Среднее без выбросов: ', np.mean(clean))
print('Медиана: ', np.median(data_steps))

Среднее с выбросами:  10111.666666666666
Среднее без выбросов:  9544.444444444445
Медиана:  9300.0


Очень большое значение (около 30 км за день) могло появиться из-за длительного похода или ошибки шагомера, например если устройство засчитывало тряску в транспорте. Два очень маленьких значения идут подряд: скорее всего, человек болел, не носил шагомер или устройство было разряжено.

Выбросы завышают среднее, а медиана к ним устойчива. Если выбросы вызваны ошибкой устройства, их следует удалить или заменить на медиану. Если это реальная активность, их можно оставить, но для оценки типичного дня лучше использовать медиану.

### Индивидуальный вопрос варианта 2
Считаем, сколько дней шаги были меньше 8000, и какой это процент от всех дней.

In [10]:
days = np.sum(data_steps < 8000)
print('Дни, когда шагов меньше 8000:', days)
print('Процент таких дней:', round(days / len(data_steps) * 100, 2), '%')

Дни, когда шагов меньше 8000: 7
Процент таких дней: 23.33 %


На маленькое количество шагов могли повлиять плохое самочувствие, выходные дни без необходимости выходить из дома, плохая погода, учёба или работа из дома, а также то, что человек не носил шагомер весь день.

### Часть 2. Работа с матрицей
Из массива `A` с помощью `reshape` собираем матрицу 3×5. Для неё выводим размерность, общее количество элементов, суммы по столбцам и по строкам, а также транспонированную матрицу.

In [11]:
A = np.array([8000, 12000, 9500, 7000, 11000, 8500, 13000, 6000, 10000, 9000, 7500, 11500, 8200, 9800,
10500])

In [12]:
matrix = A.reshape(3,5)
print(matrix)

[[ 8000 12000  9500  7000 11000]
 [ 8500 13000  6000 10000  9000]
 [ 7500 11500  8200  9800 10500]]


In [13]:
print("размерность матрицы: ", matrix.shape)
print("общее количество элементов: ", np.size(matrix))
print("сумма элементов по столбцам: ", np.sum(matrix, axis = 0))
print("сумма элементов по строкам: ", np.sum(matrix, axis = 1))
matrix_T = matrix.T
print('транспонированная матрица: ', matrix_T)

размерность матрицы:  (3, 5)
общее количество элементов:  15
сумма элементов по столбцам:  [24000 36500 23700 26800 30500]
сумма элементов по строкам:  [47500 46500 47500]
транспонированная матрица:  [[ 8000  8500  7500]
 [12000 13000 11500]
 [ 9500  6000  8200]
 [ 7000 10000  9800]
 [11000  9000 10500]]


В конце считаем показатели по всем элементам матрицы: среднее значение, стандартное отклонение, дисперсию и квартили.

In [14]:
print("Среднее:", np.mean(matrix))
print("Стандартное отклонение:", np.std(matrix, ddof=1))
print("Дисперсия:", np.var(matrix, ddof=1))
print("Квартили:", np.percentile(matrix, [25, 50, 75]))

Среднее: 9433.333333333334
Стандартное отклонение: 1958.7410923005282
Дисперсия: 3836666.6666666665
Квартили: [ 8100.  9500. 10750.]


Выбросов в массиве `A` нет, поэтому среднее и медиана близки.

## Вывод
В ходе выполнения данной практической работы были изучены основные функции библиотеки NumPy для расчёта показателей описательной статистики, научились находить выбросы с помощью межквартильного размаха и выполнять операции с матрицами.